## Data Import

In [ ]:
#Import modules
import sys
import os
print(os.getcwd())
sys.path.append('../')
sys.path.append('../modules/')
sys.path.append('../../PARAM_GEN/')
from Class_sem2dpack import *
from Stage_module import *
from houches_fb import *
from Soil_Models import Volvi, Rome
from scipy.interpolate import griddata, interp1d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim

#Load data
direct1 = "C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/OUTPUT/Preno_Volvi"
is_overburden = False
fmin, fmax = 0.01, 50
SEM = sem2dpack(direct1)
SEM.read_seismo('x')
SEM.filter_seismo(fmax=fmax, ftype='lowpass')
SEM.filter_seismo(fmin=fmin, ftype='highpass')
Time_stations = SEM.time
Vx = SEM.velocity[:,:]
XSTA, ZSTA = SEM.rcoord[:,0], SEM.rcoord[:,1]

model = Volvi


## Input Signal

In [ ]:
plot_input_signal(SEM.directory+'/inputp1', 'VF', pad=3, fmax=20, Amax=1e-2)

## Seismograms

In [ ]:
def plot_layers_seismogram(config):
    fig = plt.figure(figsize=(20,20))
    ax = fig.add_subplot(1,1,1, aspect=3)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Depth (m)")
    ax.set_title("Vx at the stations")
    ax.grid(True)

    _, layers = model()
    xmin, xmax = np.min(XSTA),np.max(XSTA)
    zmin = np.min(ZSTA)
    stations = get_stations_from_layers(layers, (xmin, xmax), zmin, XSTA, ZSTA)
    level = [0]
    for i in range(1,len(layers)):
        level.append(level[-1]-layers[i-1])
    level = np.append(level, zmin)

    if config == 'PROP':
        scale = 20
        for l in np.append(level, np.min(ZSTA)):
            ax.hlines(l/scale, Time_stations[0], Time_stations[-1], color='black', linestyles='--', lw=0.75)
        for i in range(len(stations)): 
            ax.plot(Time_stations,Vx[:,stations[i]]+ZSTA[stations[i]]/scale,label=f"Material {i+1}")
        ylabels = list(map("{:.1f}".format,level))
        ax.set_yticks(level/scale, ylabels)

    elif config == 'CONST':
        scale =4
        for i in range(len(stations)): 
            ax.plot(Time_stations,Vx[:,stations[i]] - i/scale,label=f"Material {i+1}")
        ylabels = list(map("{:.1f}".format,ZSTA[stations]))
        ax.set_yticks(-np.arange((len(stations)))/scale, ylabels)

    plt.show()

plot_layers_seismogram(config='CONST')

## Transfer Function

In [ ]:
mat_list, layers = model()
xmin, xmax = np.min(XSTA), np.max(XSTA)
zmin = np.min(ZSTA)
stations = get_stations_from_layers(layers, (xmin, xmax), zmin, XSTA, ZSTA)

# p_val = 25/46
p_val = 0.0

# Signal de sortie (output)
output_velocity = Vx[:, stations[0]]
dt_o = Time_stations[1] - Time_stations[0]

# Signal d'entrée (input)
input_signal = np.genfromtxt(open(SEM.directory + '/inputp1', 'r'))
time = input_signal[:, 0]
input_velocity = input_signal[:, 1]
dt_i = time[1] - time[0]

# Zero padding à la même taille pour aligner les fréquences
N_pad = max(len(output_velocity), len(input_velocity))
if N_pad == len(output_velocity):
    dt = dt_o
else:
    dt = dt_i

output_velocity_padded = np.pad(output_velocity, (0, N_pad - len(output_velocity)), mode='constant')
input_velocity_padded = np.pad(input_velocity, (0, N_pad - len(input_velocity)), mode='constant')

FFT1, freq1 = fourier(output_velocity_padded, dt, remove_mean=True)
FFT2, freq2 = fourier(input_velocity_padded, dt, remove_mean=True)

# Vérification que freq1 et freq2 sont identiques
if not np.allclose(freq1, freq2):
    raise ValueError("Les fréquences ne sont pas alignées après zero padding.")

# Calcul du ratio
ratio = FFT1 / FFT2

# Calcul de f0
H_star = np.sum(layers)
V_star = H_star / np.sum(layers / np.array([m.mat_dic['cs'] for m in mat_list]))
f0 = V_star / (4 * H_star)
print(f0)

# Affichage
plt.plot(freq1, ratio)
plt.vlines(f0, 0, np.max(ratio), colors='red')
plt.xlim([0.01, 7])
plt.ylim([0, 500])
plt.xscale('linear')
plt.yscale('linear')
plt.grid(True)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Spectral Amplitude (m)")
plt.title("Transfer function")
plt.show()

## Velocity Profile

In [ ]:
fig, ax = plt.subplots()

cs = np.array([m.mat_dic['cs'] for m in mat_list])
_, layers = model()
level = [0]
for i in range(1,len(layers)):
    level.append(level[-1]-layers[i-1])
level = np.append(level, zmin)

ax.grid(True)
ax.set_xlabel("S-Wave Velocity (m/s)")
ax.set_ylabel("Depth (m)")
ax.set_title("Evolution of S-Wave velocity through depth")

ax.vlines(cs[0], level[0], level[1], colors='red')
for i in range(1,level.size-1):
    ax.vlines(cs[i], level[i], level[i+1], colors='red')
    ax.hlines(level[i], cs[i-1], cs[i], colors='black', ls='dotted')
ax.vlines(cs[-1], level[-2], level[-1], colors='red')    
